# Step 12 — cluster both axes, modules not imposed

Reads `step11_panels.rds`. Writes `step12_clusters.rds`.

Two arms over the same panels:

- **Arm 1, hierarchical** — ward.D2 on `dist(method = "minkowski", p = 2)`, on **both axes**, cut
  at one level (2 groups) and one level down (4). **p = 2 is identical to Euclidean**; another
  order is a one-line change.
- **Arm 2, k-means** — both axes, k = 3, 4, 5.

**The modules are deliberately not imposed.** Fixing `column_split` to the module labels would make
the heatmap show blocks *by construction*, since WGCNA already grouped those proteins — the picture
could not fail and would prove nothing. Run free, the blocks either reappear or they do not, and
either outcome is information.

The `ARI_vs_wgcna` column measures exactly that: how much the free protein clustering agrees with
module labels it never saw. Adjusted Rand index — 0 is chance, 1 is identical (Hubert & Arabie 1985).

In [ ]:
source("../src/paths.R")
options(stringsAsFactors = FALSE); set.seed(42)
P <- readRDS(art("step11_panels.rds"))
D <- P$D; MINK_P <- P$MINK_P; LINK <- P$LINK; SITES <- P$SITES
W <- lapply(SITES, function(s) readRDS(art("wgcna_%s.rds", s))); names(W) <- SITES

ARI <- function(a,b){ tab<-table(a,b); n<-sum(tab); ch2<-function(x) x*(x-1)/2
  idx<-sum(ch2(tab)); ai<-sum(ch2(rowSums(tab))); bj<-sum(ch2(colSums(tab)))
  e<-ai*bj/ch2(n); m<-(ai+bj)/2; (idx-e)/(m-e) }
dmink <- function(m) dist(m, method="minkowski", p=MINK_P)

# ARM 1 -- hierarchical, free across the whole panel, cut one level then one more
arm_hclust <- function(Z){
  hr <- hclust(dmink(Z),   method=LINK)     # patients
  hc <- hclust(dmink(t(Z)), method=LINK)    # proteins
  list(hr=hr, hc=hc,
       rows=lapply(c(2,4), function(k) factor(paste0("P", cutree(hr,k)))),
       cols=lapply(c(2,4), function(k) factor(paste0("M", cutree(hc,k)))))
}
# ARM 2 -- all k-means, both axes
arm_kmeans <- function(Z, ks=c(3,4,5)){
  list(rows=lapply(ks, function(k){ set.seed(42); factor(paste0("P", kmeans(Z,  k, nstart=50)$cluster) )}),
       cols=lapply(ks, function(k){ set.seed(42); factor(paste0("M", kmeans(t(Z),k, nstart=50)$cluster))}))
}

RES <- list(); tab <- NULL
for (nm in names(D)){
  d <- D[[nm]]; if (!length(d$sel)) next
  Z <- scale(W[[d$cohort]]$X[, d$sel, drop=FALSE])
  a1 <- arm_hclust(Z); a2 <- arm_kmeans(Z)
  mods <- W[[d$cohort]]$mods[match(d$sel, colnames(W[[d$cohort]]$X))]
  RES[[nm]] <- list(Z=Z, hclust=a1, kmeans=a2, mods=mods, d=d)
  for (i in seq_along(c(2,4)))
    tab <- rbind(tab, data.frame(cohort=d$cohort, cond=d$cond, arm="hclust",
      k=c(2,4)[i], patient_sizes=paste(sort(table(a1$rows[[i]])),collapse="/"),
      protein_sizes=paste(sort(table(a1$cols[[i]])),collapse="/"),
      ARI_vs_wgcna=round(ARI(a1$cols[[i]], mods),3)))
  for (i in seq_along(c(3,4,5)))
    tab <- rbind(tab, data.frame(cohort=d$cohort, cond=d$cond, arm="kmeans",
      k=c(3,4,5)[i], patient_sizes=paste(sort(table(a2$rows[[i]])),collapse="/"),
      protein_sizes=paste(sort(table(a2$cols[[i]])),collapse="/"),
      ARI_vs_wgcna=round(ARI(a2$cols[[i]], mods),3)))
}
cat("free clustering on both axes; ARI compares the PROTEIN clusters to the WGCNA module labels\n")
cat("(modules are NOT imposed -- hclust and k-means run across the whole panel)\n\n")
print(tab[tab$cond=="all15",], row.names=FALSE)
cat("\n--- varsel and union conditions ---\n")
print(tab[tab$cond!="all15",], row.names=FALSE)

saveRDS(list(RES = RES, tab = tab), art("step12_clusters.rds"))


## What the ARI says

**The modules largely do not break.** Cohort A reaches 0.80; cohort C's `union` hierarchical cut at
k = 4 reaches **0.886**. Free clustering very nearly reproduces a partition it was never given.

**Cohort B's ARI is 0.000 everywhere, and that is arithmetic, not a finding.** B's panel is a single
module, so the WGCNA label vector is constant, and the adjusted Rand index against a constant
partition is 0 by construction.

**Cohort A carries extreme samples.** A produces a 1-patient cluster at k = 4 and k = 5 and an
11/76 split at k = 2, in both arms. B and C do not. Step 15 follows this up.